# Lesson 5: Human in the Loop

In [1]:
import subprocess
import sys
from pathlib import Path

print("Kernel Python:", sys.executable)

_here = Path.cwd()
_req = _here / "requirements-lesson-05.txt"
if not _req.exists():
    _req = _here.parent / "requirements-lesson-05.txt"

if _req.exists():
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", str(_req)])
    print("Installed from", _req)
else:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        "langgraph", "langgraph-checkpoint-sqlite", "langchain-core", "langchain-openai",
        "langchain-community", "tavily-python", "python-dotenv", "httpx", "ipykernel",
    ])
    print("Installed packages via pip (requirements file not found).")


Kernel Python: /opt/anaconda3/bin/python
Installed from /Users/mkenne16/Documents/LangGraph/requirements-lesson-05.txt


## Custom message reducer

Earlier lessons annotated `messages` with `operator.add` so new messages are always **appended**.

Here we use a **custom reducer** that **replaces** messages that share the same `id` and **appends** otherwise — useful when you edit or replay state.


In [2]:
from pathlib import Path

from dotenv import load_dotenv

_here = Path.cwd()
for _p in (_here / ".env", _here.parent / ".env"):
    if _p.exists():
        load_dotenv(_p)
        break
else:
    load_dotenv()

import operator
import sqlite3
from typing import Annotated, TypedDict
from uuid import uuid4

from langchain_community.tools.tavily_search import TavilySearchResults
from langchain_core.messages import AnyMessage, HumanMessage, SystemMessage, ToolMessage
from langchain_openai import ChatOpenAI
from langgraph.checkpoint.sqlite import SqliteSaver
from langgraph.graph import END, StateGraph


def reduce_messages(left: list[AnyMessage], right: list[AnyMessage]) -> list[AnyMessage]:
    # assign ids to messages that don't have them
    for message in right:
        if not message.id:
            message.id = str(uuid4())
    merged = left.copy()
    for message in right:
        for i, existing in enumerate(merged):
            if existing.id == message.id:
                merged[i] = message
                break
        else:
            merged.append(message)
    return merged


class AgentState(TypedDict):
    messages: Annotated[list[AnyMessage], reduce_messages]


tool = TavilySearchResults(max_results=2)


class Agent:
    def __init__(self, model, tools, system="", checkpointer=None):
        self.system = system
        graph = StateGraph(AgentState)
        graph.add_node("llm", self.call_openai)
        graph.add_node("action", self.take_action)
        graph.add_conditional_edges("llm", self.exists_action, {True: "action", False: END})
        graph.add_edge("action", "llm")
        graph.set_entry_point("llm")
        self.graph = graph.compile(
            checkpointer=checkpointer,
            interrupt_before=["action"],
        )
        self.tools = {t.name: t for t in tools}
        self.model = model.bind_tools(tools)

    def call_openai(self, state: AgentState):
        messages = state["messages"]
        if self.system:
            messages = [SystemMessage(content=self.system)] + messages
        message = self.model.invoke(messages)
        return {"messages": [message]}

    def exists_action(self, state: AgentState):
        print(state)
        result = state["messages"][-1]
        return len(result.tool_calls) > 0

    def take_action(self, state: AgentState):
        tool_calls = state["messages"][-1].tool_calls
        results = []
        for t in tool_calls:
            print(f"Calling: {t}")
            result = self.tools[t["name"]].invoke(t["args"])
            results.append(ToolMessage(tool_call_id=t["id"], name=t["name"], content=str(result)))
        print("Back to the model!")
        return {"messages": results}


prompt = """You are a smart research assistant. Use the search engine to look up information. \
You are allowed to make multiple calls (either together or in sequence). \
Only look up information when you are sure of what you want. \
If you need to look up some information before asking a follow up question, you are allowed to do that!
"""

model = ChatOpenAI(model="gpt-3.5-turbo")

_sqlite_conn = sqlite3.connect(":memory:", check_same_thread=False)
memory = SqliteSaver(_sqlite_conn)

abot = Agent(model, [tool], system=prompt, checkpointer=memory)


/var/folders/2p/x0g54f5s6_v5t47f_25hgyww0000gq/T/ipykernel_98522/3162330258.py:45: LangChainDeprecationWarning: The class `TavilySearchResults` was deprecated in LangChain 0.3.25 and will be removed in 1.0. An updated version of the class exists in the `langchain-tavily package and should be used instead. To use it run `pip install -U `langchain-tavily` and import as `from `langchain_tavily import TavilySearch``.
  tool = TavilySearchResults(max_results=2)


## Manual human approval

`interrupt_before=["action"]` pauses **before** the tool node runs. Inspect `get_state()`, then resume with `stream(None, config)`.


In [3]:
messages = [HumanMessage(content="Whats the weather in SF?")]
thread = {"configurable": {"thread_id": "1"}}
for event in abot.graph.stream({"messages": messages}, thread):
    for v in event.values():
        print(v)

abot.graph.get_state(thread)


{'messages': [HumanMessage(content='Whats the weather in SF?', additional_kwargs={}, response_metadata={}, id='01133ca9-7247-4d3c-8028-92c88b9a8805'), AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 22, 'prompt_tokens': 152, 'total_tokens': 174, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-DODX5NMzt2h5wLTyAF0Yjp8I1k5uV', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019d323b-a322-7923-9181-7d1e9bd81f2b-0', tool_calls=[{'name': 'tavily_search_results_json', 'args': {'query': 'current weather in San Francisco'}, 'id': 'call_7T8HaVujsUdNcTxPYfL98rt8', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'inp

StateSnapshot(values={'messages': [HumanMessage(content='Whats the weather in SF?', additional_kwargs={}, response_metadata={}, id='01133ca9-7247-4d3c-8028-92c88b9a8805'), AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 22, 'prompt_tokens': 152, 'total_tokens': 174, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-DODX5NMzt2h5wLTyAF0Yjp8I1k5uV', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019d323b-a322-7923-9181-7d1e9bd81f2b-0', tool_calls=[{'name': 'tavily_search_results_json', 'args': {'query': 'current weather in San Francisco'}, 'id': 'call_7T8HaVujsUdNcTxPYfL98rt8', 'type': 'tool_call'}], invalid_tool_calls=[],

In [4]:
abot.graph.get_state(thread).next


('action',)

## Continue after interrupt

Pass **`None`** as input to continue from the saved checkpoint.


In [5]:
for event in abot.graph.stream(None, thread):
    for v in event.values():
        print(v)

abot.graph.get_state(thread)


Calling: {'name': 'tavily_search_results_json', 'args': {'query': 'current weather in San Francisco'}, 'id': 'call_7T8HaVujsUdNcTxPYfL98rt8', 'type': 'tool_call'}
Back to the model!
{'messages': [ToolMessage(content='[{\'title\': \'San Francisco, CA Weather Forecast - AccuWeather\', \'url\': \'https://www.accuweather.com/en/us/san-francisco/94103/weather-forecast/347629\', \'content\': "rain drop\\n\\nrain drop\\n\\nrain drop\\n\\nrain drop\\n\\nrain drop\\n\\nrain drop\\n\\nrain drop\\n\\nrain drop\\n\\nrain drop\\n\\nrain drop\\n\\nrain drop\\n\\nrain drop\\n\\n## 10-Day Weather Forecast\\n\\nTonight\\n\\n3/27\\n\\nAreas of low clouds\\n\\nSat\\n\\n3/28\\n\\nPartly sunny and cooler\\n\\nNight: Partly cloudy\\n\\nSun\\n\\n3/29\\n\\nPartly sunny and pleasant\\n\\nMainly clear\\n\\nMon\\n\\n3/30\\n\\nCooler with clouds and sun\\n\\nPartly cloudy\\n\\nTue\\n\\n3/31\\n\\nMostly cloudy\\n\\nPartly cloudy\\n\\nWed\\n\\n4/1\\n\\nTimes of clouds and sun\\n\\nEarly showers; mostly cloudy\\n\\n

StateSnapshot(values={'messages': [HumanMessage(content='Whats the weather in SF?', additional_kwargs={}, response_metadata={}, id='01133ca9-7247-4d3c-8028-92c88b9a8805'), AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 22, 'prompt_tokens': 152, 'total_tokens': 174, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-DODX5NMzt2h5wLTyAF0Yjp8I1k5uV', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019d323b-a322-7923-9181-7d1e9bd81f2b-0', tool_calls=[{'name': 'tavily_search_results_json', 'args': {'query': 'current weather in San Francisco'}, 'id': 'call_7T8HaVujsUdNcTxPYfL98rt8', 'type': 'tool_call'}], invalid_tool_calls=[],

In [6]:
abot.graph.get_state(thread).next


()

## New thread + interactive loop

A new `thread_id` starts a fresh conversation. The loop calls `input()` — type **`y`** to approve each tool step (Jupyter will prompt in the cell output).


In [ ]:
messages = [HumanMessage(content="Whats the weather in LA?")]
thread = {"configurable": {"thread_id": "2"}}
for event in abot.graph.stream({"messages": messages}, thread):
    for v in event.values():
        print(v)

while abot.graph.get_state(thread).next:
    print("\n", abot.graph.get_state(thread), "\n")
    _input = input("proceed?")
    if _input != "y":
        print("aborting")
        break
    for event in abot.graph.stream(None, thread):
        for v in event.values():
            print(v)


## Modify state

Run until the interrupt, then change the pending tool call (e.g. different query) and call **`update_state`**. Keep the same tool-call **`id`** so the model and checkpoint stay consistent.


In [ ]:
messages = [HumanMessage(content="Whats the weather in LA?")]
thread = {"configurable": {"thread_id": "3"}}
for event in abot.graph.stream({"messages": messages}, thread):
    for v in event.values():
        print(v)

abot.graph.get_state(thread)


{'messages': [HumanMessage(content='Whats the weather in LA?', additional_kwargs={}, response_metadata={}, id='87c3bde6-cf01-43bb-a43a-fa10a3046c0c'), AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 21, 'prompt_tokens': 152, 'total_tokens': 173, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-DNONx7co0GXgqU8GyvzrUKL5hEW7k', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019d2683-9a90-7ca1-b92d-615d165070bf-0', tool_calls=[{'name': 'tavily_search_results_json', 'args': {'query': 'weather in Los Angeles'}, 'id': 'call_tBJQY5sPJiuKTMZN2yldeiuG', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens'

StateSnapshot(values={'messages': [HumanMessage(content='Whats the weather in LA?', additional_kwargs={}, response_metadata={}, id='87c3bde6-cf01-43bb-a43a-fa10a3046c0c'), AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 21, 'prompt_tokens': 152, 'total_tokens': 173, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-DNONx7co0GXgqU8GyvzrUKL5hEW7k', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019d2683-9a90-7ca1-b92d-615d165070bf-0', tool_calls=[{'name': 'tavily_search_results_json', 'args': {'query': 'weather in Los Angeles'}, 'id': 'call_tBJQY5sPJiuKTMZN2yldeiuG', 'type': 'tool_call'}], invalid_tool_calls=[], usage_met

In [ ]:
current_values = abot.graph.get_state(thread)
current_values.values["messages"][-1]


AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 21, 'prompt_tokens': 152, 'total_tokens': 173, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-DNONx7co0GXgqU8GyvzrUKL5hEW7k', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019d2683-9a90-7ca1-b92d-615d165070bf-0', tool_calls=[{'name': 'tavily_search_results_json', 'args': {'query': 'weather in Los Angeles'}, 'id': 'call_tBJQY5sPJiuKTMZN2yldeiuG', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 152, 'output_tokens': 21, 'total_tokens': 173, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning'

In [ ]:
current_values.values["messages"][-1].tool_calls


[{'name': 'tavily_search_results_json',
  'args': {'query': 'weather in Los Angeles'},
  'id': 'call_tBJQY5sPJiuKTMZN2yldeiuG',
  'type': 'tool_call'}]

In [ ]:
_id = current_values.values["messages"][-1].tool_calls[0]["id"]
current_values.values["messages"][-1].tool_calls = [
    {
        "name": "tavily_search_results_json",
        "args": {"query": "current weather in Louisiana"},
        "id": _id,
    }
]

abot.graph.update_state(thread, current_values.values)

abot.graph.get_state(thread)


{'messages': [HumanMessage(content='Whats the weather in LA?', additional_kwargs={}, response_metadata={}, id='87c3bde6-cf01-43bb-a43a-fa10a3046c0c'), AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 21, 'prompt_tokens': 152, 'total_tokens': 173, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-DNONx7co0GXgqU8GyvzrUKL5hEW7k', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019d2683-9a90-7ca1-b92d-615d165070bf-0', tool_calls=[{'name': 'tavily_search_results_json', 'args': {'query': 'current weather in Louisiana'}, 'id': 'call_tBJQY5sPJiuKTMZN2yldeiuG'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 152, 'output_

StateSnapshot(values={'messages': [HumanMessage(content='Whats the weather in LA?', additional_kwargs={}, response_metadata={}, id='87c3bde6-cf01-43bb-a43a-fa10a3046c0c'), AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 21, 'prompt_tokens': 152, 'total_tokens': 173, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-DNONx7co0GXgqU8GyvzrUKL5hEW7k', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019d2683-9a90-7ca1-b92d-615d165070bf-0', tool_calls=[{'name': 'tavily_search_results_json', 'args': {'query': 'current weather in Louisiana'}, 'id': 'call_tBJQY5sPJiuKTMZN2yldeiuG', 'type': 'tool_call'}], invalid_tool_calls=[], usa

In [ ]:
for event in abot.graph.stream(None, thread):
    for v in event.values():
        print(v)


Calling: {'name': 'tavily_search_results_json', 'args': {'query': 'current weather in Louisiana'}, 'id': 'call_tBJQY5sPJiuKTMZN2yldeiuG', 'type': 'tool_call'}
Back to the model!
{'messages': [ToolMessage(content='[{\'title\': \'Weather Forecast for Louisiana for Wednesday 25 March\', \'url\': \'https://www.metcheck.com/WEATHER/dayforecast.asp?location=Louisiana&dateFor=25/03/2026&locationID=413448&lat=10.132990&lon=-83.529350\', \'content\': \'Cloudy Later With Max Temp Of 24°c And Min Temp Of 13°c 24°c 13°c |  Wed) | [...] UV (UV Index) - This is the UV Index from 0 to 10 based on the cloud cover and strength of sunshine [...] Feels (2 metre RealFeel Temperature) - This is what the temperature will feel like based on wind speed, temperature and humidity  \\nRain (Rainfall) - This is the total amount of rainfall or precipitation expected from stratiform clouds in the forecast period.  \\nCloud (Total Cloud Cover) - This is the average cloud cover across the region in the 1 or 3hr perio

## Time travel

Iterate **`get_state_history`** (most recent first). Pick an older snapshot and resume with **`stream(None, snapshot.config)`**.

**Index tip:** In the filmed course, `states[-1]` pointed at a certain checkpoint. With more checkpoints per run, you often need an offset such as **`states[-3]`** (accounts for extra entries like `__start__` / first transitions). **Inspect your printed history** and choose the snapshot you want — the right index can vary by LangGraph version.


In [ ]:
states = []
for state in abot.graph.get_state_history(thread):
    print(state)
    print("--")
    states.append(state)


StateSnapshot(values={'messages': [HumanMessage(content='Whats the weather in LA?', additional_kwargs={}, response_metadata={}, id='87c3bde6-cf01-43bb-a43a-fa10a3046c0c'), AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 21, 'prompt_tokens': 152, 'total_tokens': 173, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-DNONx7co0GXgqU8GyvzrUKL5hEW7k', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019d2683-9a90-7ca1-b92d-615d165070bf-0', tool_calls=[{'name': 'tavily_search_results_json', 'args': {'query': 'current weather in Louisiana'}, 'id': 'call_tBJQY5sPJiuKTMZN2yldeiuG', 'type': 'tool_call'}], invalid_tool_calls=[], usa

In [ ]:
to_replay = states[-3]
to_replay


StateSnapshot(values={'messages': [HumanMessage(content='Whats the weather in LA?', additional_kwargs={}, response_metadata={}, id='87c3bde6-cf01-43bb-a43a-fa10a3046c0c'), AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 21, 'prompt_tokens': 152, 'total_tokens': 173, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-DNONx7co0GXgqU8GyvzrUKL5hEW7k', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019d2683-9a90-7ca1-b92d-615d165070bf-0', tool_calls=[{'name': 'tavily_search_results_json', 'args': {'query': 'weather in Los Angeles'}, 'id': 'call_tBJQY5sPJiuKTMZN2yldeiuG', 'type': 'tool_call'}], invalid_tool_calls=[], usage_met

In [ ]:
for event in abot.graph.stream(None, to_replay.config):
    for k, v in event.items():
        print(v)


Calling: {'name': 'tavily_search_results_json', 'args': {'query': 'weather in Los Angeles'}, 'id': 'call_tBJQY5sPJiuKTMZN2yldeiuG', 'type': 'tool_call'}
Back to the model!
{'messages': [ToolMessage(content="[{'title': 'Los Angeles weather in March 2026 | Weather25.com', 'url': 'https://www.weather25.com/north-america/usa/california/los-angeles?page=month&month=March', 'content': '## The average weather in Los Angeles in March\\n\\nThe temperatures in Los Angeles in March are comfortable with low of 12°C and and high up to 22°C.\\n\\nThe wather in Los Angeles in March can vary between cold and nice weather days. Expect a few rainy days but usually not more than 3.\\n\\nOur weather forecast can give you a great sense of what weather to expect in Los Angeles in March 2026.\\n\\nIf you’re planning to visit Los Angeles in the near future, we highly recommend that you review the 14 day weather forecast for Los Angeles before you arrive.\\n\\nImage 22: Temperatures\\n\\nTemperatures\\n\\n22° 

## Go back in time and edit

Mutate `to_replay.values`, then **`update_state(to_replay.config, ...)`** to branch. Resume from the returned config.


In [ ]:
to_replay
_id = to_replay.values["messages"][-1].tool_calls[0]["id"]
to_replay.values["messages"][-1].tool_calls = [
    {
        "name": "tavily_search_results_json",
        "args": {"query": "current weather in LA, accuweather"},
        "id": _id,
    }
]
branch_state = abot.graph.update_state(to_replay.config, to_replay.values)
for event in abot.graph.stream(None, branch_state):
    for k, v in event.items():
        if k != "__end__":
            print(v)


{'messages': [HumanMessage(content='Whats the weather in LA?', additional_kwargs={}, response_metadata={}, id='87c3bde6-cf01-43bb-a43a-fa10a3046c0c'), AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 21, 'prompt_tokens': 152, 'total_tokens': 173, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-DNONx7co0GXgqU8GyvzrUKL5hEW7k', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019d2683-9a90-7ca1-b92d-615d165070bf-0', tool_calls=[{'name': 'tavily_search_results_json', 'args': {'query': 'current weather in LA, accuweather'}, 'id': 'call_tBJQY5sPJiuKTMZN2yldeiuG'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 152, 'o

## Add a message at a point in time

Use **`as_node="action"`** so the graph treats the update as if the action node produced it, which fixes up **`next`** for following edges.


In [ ]:
to_replay
_id = to_replay.values["messages"][-1].tool_calls[0]["id"]
state_update = {
    "messages": [
        ToolMessage(
            tool_call_id=_id,
            name="tavily_search_results_json",
            content="54 degree celcius",
        )
    ]
}
branch_and_add = abot.graph.update_state(to_replay.config, state_update, as_node="action")
for event in abot.graph.stream(None, branch_and_add):
    for k, v in event.items():
        print(v)


{'messages': [HumanMessage(content='Whats the weather in LA?', additional_kwargs={}, response_metadata={}, id='87c3bde6-cf01-43bb-a43a-fa10a3046c0c'), AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 21, 'prompt_tokens': 152, 'total_tokens': 173, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-DNONx7co0GXgqU8GyvzrUKL5hEW7k', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019d2683-9a90-7ca1-b92d-615d165070bf-0', tool_calls=[{'name': 'tavily_search_results_json', 'args': {'query': 'weather in Los Angeles'}, 'id': 'call_tBJQY5sPJiuKTMZN2yldeiuG', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens'

## Extra practice: small graph

Two-node loop with a counter. Uses **separate** checkpoint objects (`toy_memory`, `toy_graph`) so you do not overwrite the main agent demo.

**Graph image:** `draw_png()` may need extra system packages (e.g. Graphviz / Mermaid rendering). If it errors, skip that cell or install the dependency for your OS.


In [ ]:
from langgraph.graph import END, StateGraph
from typing import Annotated, TypedDict

import operator
import sqlite3
from langgraph.checkpoint.sqlite import SqliteSaver


class ToyState(TypedDict):
    lnode: str
    scratch: str
    count: Annotated[int, operator.add]


def node1(state: ToyState):
    print(f"node1, count:{state['count']}")
    return {"lnode": "node_1", "count": 1}


def node2(state: ToyState):
    print(f"node2, count:{state['count']}")
    return {"lnode": "node_2", "count": 1}


def should_continue(state):
    return state["count"] < 3


builder = StateGraph(ToyState)
builder.add_node("Node1", node1)
builder.add_node("Node2", node2)
builder.add_edge("Node1", "Node2")
builder.add_conditional_edges("Node2", should_continue, {True: "Node1", False: END})
builder.set_entry_point("Node1")

_toy_sqlite = sqlite3.connect(":memory:", check_same_thread=False)
toy_memory = SqliteSaver(_toy_sqlite)
toy_graph = builder.compile(checkpointer=toy_memory)


In [ ]:
thread = {"configurable": {"thread_id": str(1)}}
toy_graph.invoke({"count": 0, "scratch": "hi"}, thread)


node1, count:0
node2, count:1
node1, count:2
node2, count:3


{'lnode': 'node_2', 'scratch': 'hi', 'count': 4}

In [ ]:
toy_graph.get_state(thread)


StateSnapshot(values={'lnode': 'node_2', 'scratch': 'hi', 'count': 4}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f128829-2d27-6f8c-8004-5ba3a1127379'}}, metadata={'source': 'loop', 'step': 4, 'parents': {}}, created_at='2026-03-25T19:41:09.932423+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f128829-2d26-66d2-8003-79fb8ae2996f'}}, tasks=(), interrupts=())

In [ ]:
for state in toy_graph.get_state_history(thread):
    print(state, "\n")


StateSnapshot(values={'lnode': 'node_2', 'scratch': 'hi', 'count': 4}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f128829-2d27-6f8c-8004-5ba3a1127379'}}, metadata={'source': 'loop', 'step': 4, 'parents': {}}, created_at='2026-03-25T19:41:09.932423+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f128829-2d26-66d2-8003-79fb8ae2996f'}}, tasks=(), interrupts=()) 

StateSnapshot(values={'lnode': 'node_1', 'scratch': 'hi', 'count': 3}, next=('Node2',), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f128829-2d26-66d2-8003-79fb8ae2996f'}}, metadata={'source': 'loop', 'step': 3, 'parents': {}}, created_at='2026-03-25T19:41:09.931787+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f128829-2d24-6512-8002-3ae59bbd12df'}}, tasks=(PregelTask(id='8daaa4d0-aa1b-2783-2462-4ccfd3510dba', name='Node2', path=('__pregel_pull

In [ ]:
toy_cfgs = []
for state in toy_graph.get_state_history(thread):
    toy_cfgs.append(state.config)
    print(state.config, state.values["count"])


{'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f128829-2d27-6f8c-8004-5ba3a1127379'}} 4
{'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f128829-2d26-66d2-8003-79fb8ae2996f'}} 3
{'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f128829-2d24-6512-8002-3ae59bbd12df'}} 2
{'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f128829-2d22-680c-8001-902d95fbf897'}} 1
{'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f128829-2d20-6f02-8000-e26854bada16'}} 0
{'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f128829-2d1e-6dec-bfff-9b6308286b0c'}} 0


In [ ]:
toy_cfgs[-3]


{'configurable': {'thread_id': '1',
  'checkpoint_ns': '',
  'checkpoint_id': '1f128829-2d22-680c-8001-902d95fbf897'}}

In [ ]:
toy_graph.get_state(toy_cfgs[-3])


StateSnapshot(values={'lnode': 'node_1', 'scratch': 'hi', 'count': 1}, next=('Node2',), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f128829-2d22-680c-8001-902d95fbf897'}}, metadata={'source': 'loop', 'step': 1, 'parents': {}}, created_at='2026-03-25T19:41:09.930183+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f128829-2d20-6f02-8000-e26854bada16'}}, tasks=(PregelTask(id='e64e710f-18ba-52ac-9063-1dfe8b273a01', name='Node2', path=('__pregel_pull', 'Node2'), error=None, interrupts=(), state=None, result={'lnode': 'node_2', 'count': 1}),), interrupts=())

In [ ]:
toy_graph.invoke(None, toy_cfgs[-3])


node2, count:1
node1, count:2
node2, count:3


{'lnode': 'node_2', 'scratch': 'hi', 'count': 4}

In [ ]:
thread = {"configurable": {"thread_id": str(1)}}
for state in toy_graph.get_state_history(thread):
    print(state.config, state.values["count"])


{'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f128829-2d98-6048-8004-7497770f8ada'}} 4
{'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f128829-2d96-6d7e-8003-9b3c8d936de3'}} 3
{'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f128829-2d94-6d1c-8002-68d205cabb5b'}} 2
{'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f128829-2d27-6f8c-8004-5ba3a1127379'}} 4
{'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f128829-2d26-66d2-8003-79fb8ae2996f'}} 3
{'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f128829-2d24-6512-8002-3ae59bbd12df'}} 2
{'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f128829-2d22-680c-8001-902d95fbf897'}} 1
{'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f128829-2d20-6f02-8000-e26854bada16'}} 0
{'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkp

In [ ]:
thread = {"configurable": {"thread_id": str(1)}}
for state in toy_graph.get_state_history(thread):
    print(state, "\n")


StateSnapshot(values={'lnode': 'node_2', 'scratch': 'hi', 'count': 4}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f128829-2d98-6048-8004-7497770f8ada'}}, metadata={'source': 'loop', 'step': 4, 'parents': {}}, created_at='2026-03-25T19:41:09.978317+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f128829-2d96-6d7e-8003-9b3c8d936de3'}}, tasks=(), interrupts=()) 

StateSnapshot(values={'lnode': 'node_1', 'scratch': 'hi', 'count': 3}, next=('Node2',), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f128829-2d96-6d7e-8003-9b3c8d936de3'}}, metadata={'source': 'loop', 'step': 3, 'parents': {}}, created_at='2026-03-25T19:41:09.977835+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f128829-2d94-6d1c-8002-68d205cabb5b'}}, tasks=(PregelTask(id='913f9bd4-f272-1251-9e3f-668a8f5033b8', name='Node2', path=('__pregel_pull

In [ ]:
thread2 = {"configurable": {"thread_id": str(2)}}
toy_graph.invoke({"count": 0, "scratch": "hi"}, thread2)


node1, count:0
node2, count:1
node1, count:2
node2, count:3


{'lnode': 'node_2', 'scratch': 'hi', 'count': 4}

In [ ]:
from IPython.display import Image, display

try:
    display(Image(toy_graph.get_graph().draw_png()))
except Exception as e:
    print("Could not draw PNG:", e)


Could not draw PNG: Install pygraphviz to draw graphs: `pip install pygraphviz`.


In [ ]:
states2 = []
for state in toy_graph.get_state_history(thread2):
    states2.append(state.config)
    print(state.config, state.values["count"])


{'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f128829-2de0-6b68-8004-61b887046e1d'}} 4
{'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f128829-2ddf-6538-8003-0f0b6fe341e6'}} 3
{'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f128829-2ddd-603a-8002-c18da1750c39'}} 2
{'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f128829-2dd8-6aee-8001-0027c8ed0feb'}} 1
{'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f128829-2dd6-6c9e-8000-07ca047ac63f'}} 0
{'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f128829-2dd5-659c-bfff-d803e843e52b'}} 0


In [ ]:
save_state = toy_graph.get_state(states2[-3])
save_state


StateSnapshot(values={'lnode': 'node_1', 'scratch': 'hi', 'count': 1}, next=('Node2',), config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f128829-2dd8-6aee-8001-0027c8ed0feb'}}, metadata={'source': 'loop', 'step': 1, 'parents': {}}, created_at='2026-03-25T19:41:10.004801+00:00', parent_config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f128829-2dd6-6c9e-8000-07ca047ac63f'}}, tasks=(PregelTask(id='56952457-b7bf-bb0b-7abf-28f09e26bdec', name='Node2', path=('__pregel_pull', 'Node2'), error=None, interrupts=(), state=None, result={'lnode': 'node_2', 'count': 1}),), interrupts=())

In [ ]:
save_state.values["count"] = -3
save_state.values["scratch"] = "hello"
save_state


StateSnapshot(values={'lnode': 'node_1', 'scratch': 'hello', 'count': -3}, next=('Node2',), config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f128829-2dd8-6aee-8001-0027c8ed0feb'}}, metadata={'source': 'loop', 'step': 1, 'parents': {}}, created_at='2026-03-25T19:41:10.004801+00:00', parent_config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f128829-2dd6-6c9e-8000-07ca047ac63f'}}, tasks=(PregelTask(id='56952457-b7bf-bb0b-7abf-28f09e26bdec', name='Node2', path=('__pregel_pull', 'Node2'), error=None, interrupts=(), state=None, result={'lnode': 'node_2', 'count': 1}),), interrupts=())

In [ ]:
toy_graph.update_state(thread2, save_state.values)

for i, state in enumerate(toy_graph.get_state_history(thread2)):
    if i >= 3:
        break
    print(state, "\n")


StateSnapshot(values={'lnode': 'node_1', 'scratch': 'hello', 'count': 1}, next=('Node1',), config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f128829-2e5e-63b0-8005-d05b87b49c17'}}, metadata={'source': 'update', 'step': 5, 'parents': {}}, created_at='2026-03-25T19:41:10.059498+00:00', parent_config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f128829-2de0-6b68-8004-61b887046e1d'}}, tasks=(PregelTask(id='dc026607-f24a-8e39-677e-96db4c60867e', name='Node1', path=('__pregel_pull', 'Node1'), error=None, interrupts=(), state=None, result=None),), interrupts=()) 

StateSnapshot(values={'lnode': 'node_2', 'scratch': 'hi', 'count': 4}, next=(), config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f128829-2de0-6b68-8004-61b887046e1d'}}, metadata={'source': 'loop', 'step': 4, 'parents': {}}, created_at='2026-03-25T19:41:10.008093+00:00', parent_config={'configurable': {'thread_id': '2', 'checkpoint_ns': 

In [ ]:
toy_graph.update_state(thread2, save_state.values, as_node="Node1")

for i, state in enumerate(toy_graph.get_state_history(thread2)):
    if i >= 3:
        break
    print(state, "\n")


StateSnapshot(values={'lnode': 'node_1', 'scratch': 'hello', 'count': -2}, next=('Node2',), config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f128829-2e71-60dc-8006-a531a8341bd4'}}, metadata={'source': 'update', 'step': 6, 'parents': {}}, created_at='2026-03-25T19:41:10.067210+00:00', parent_config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f128829-2e5e-63b0-8005-d05b87b49c17'}}, tasks=(PregelTask(id='3054d26a-9c2d-6631-3036-15523263f341', name='Node2', path=('__pregel_pull', 'Node2'), error=None, interrupts=(), state=None, result=None),), interrupts=()) 

StateSnapshot(values={'lnode': 'node_1', 'scratch': 'hello', 'count': 1}, next=('Node1',), config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f128829-2e5e-63b0-8005-d05b87b49c17'}}, metadata={'source': 'update', 'step': 5, 'parents': {}}, created_at='2026-03-25T19:41:10.059498+00:00', parent_config={'configurable': {'thread_id': '2', 'ch

In [ ]:
toy_graph.invoke(None, thread2)


node2, count:-2
node1, count:-1
node2, count:0
node1, count:1
node2, count:2


{'lnode': 'node_2', 'scratch': 'hello', 'count': 3}

In [ ]:
for state in toy_graph.get_state_history(thread2):
    print(state, "\n")


StateSnapshot(values={'lnode': 'node_2', 'scratch': 'hello', 'count': 3}, next=(), config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f128829-2e8f-6582-800b-c7759e02e1eb'}}, metadata={'source': 'loop', 'step': 11, 'parents': {}}, created_at='2026-03-25T19:41:10.079620+00:00', parent_config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f128829-2e8d-6e08-800a-0135c50af6d5'}}, tasks=(), interrupts=()) 

StateSnapshot(values={'lnode': 'node_1', 'scratch': 'hello', 'count': 2}, next=('Node2',), config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f128829-2e8d-6e08-800a-0135c50af6d5'}}, metadata={'source': 'loop', 'step': 10, 'parents': {}}, created_at='2026-03-25T19:41:10.079017+00:00', parent_config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f128829-2e8a-6e1a-8009-2b29d05d3191'}}, tasks=(PregelTask(id='01ed9097-f499-54e9-2743-d2921d03cdef', name='Node2', path=('__pre